<h1>RAZ Systems </h1>

## Setting up SendGrid

Please visit Sendgrid at: https://sendgrid.com/

Setting up an account is free. Once created:

Settings >> API Keys >> Create API Key, then add to your `.env`:

`SENDGRID_API_KEY=xxxx`

Then go to Settings >> Sender Authentication >> Verify a Single Sender and verify your email address.

In [2]:
print('s')

s


In [3]:
# --- Imports ---

from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [4]:
load_dotenv(override=True)

True

In [5]:
# --- Verify email is working before running the full agent pipeline ---

def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("imranlatifclouds@gmail.com")  # Change to your verified sender
    to_email = To("imranlatifclouds@gmail.com")        # Change to your recipient
    content = Content("text/plain", "SendGrid test - HNW outreach pipeline ready.")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)  # 202 = success

send_test_email()

202


### Did you receive the test email?

A **202** means you're good to go!

#### Certificate error
If you get `SSL: CERTIFICATE_VERIFY_FAILED`:
```python
# Run in terminal first: uv pip install --upgrade certifi
import certifi, os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

## Part 1a — Single Agent, Streamed Output

The simplest starting point: one advisor, one prompt, output printed token-by-token as it arrives.

**New concept:** `Runner.run_streamed()` — lets you see the response being built in real time, useful for long outputs.

In [7]:
# --- Three advisor personas for HNW outreach ---
# Same goal, different tone — lets us find the most effective approach.

instructions1 = (
    "You are a Senior Private Banker at Apex Capital, an investment bank serving ultra-high net worth individuals. "
    "You write formal, authoritative outreach emails that emphasise capital preservation, "
    "bespoke portfolio management, and exclusive access to private market opportunities."
)

instructions2 = (
    "You are a Relationship Manager at Apex Capital, an investment bank serving ultra-high net worth individuals. "
    "You write warm, confident outreach emails that build trust quickly, "
    "referencing current market conditions and the client's likely financial goals."
)

instructions3 = (
    "You are a Managing Director at Apex Capital, an investment bank serving ultra-high net worth individuals. "
    "You write short, direct outreach emails - two or three paragraphs maximum - "
    "that lead with a compelling insight and a clear call to action."
)

In [8]:
# --- Create the three advisor agents ---

advisor_agent1 = Agent(
    name="Senior Private Banker",
    instructions=instructions1,
    model="gpt-4o-mini",
)

advisor_agent2 = Agent(
    name="Relationship Manager",
    instructions=instructions2,
    model="gpt-4o-mini",
)

advisor_agent3 = Agent(
    name="Managing Director",
    instructions=instructions3,
    model="gpt-4o-mini",
)

In [9]:
# --- Stream a single draft to see what one advisor produces ---
# Runner.run_streamed() prints tokens as they arrive.
import asyncio

# Wrap in a task so Jupyter's loop can properly cancel it on interrupt
async def run():
    result = Runner.run_streamed(advisor_agent1, input="Write a HNW client outreach email")
    async for event in result.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)

task = asyncio.ensure_future(run())
await task

Subject: Exclusive Opportunities for Your Investment Strategy

Dear [Client's Name],

I hope this message finds you well. At Apex Capital, we deeply value the trust you place in us to manage your financial legacy, and I want to take this opportunity to share some insights that may be of interest to you.

In today’s evolving economic landscape, capital preservation remains paramount, especially for ultra-high net worth individuals. Our bespoke portfolio management services are designed to navigate market fluctuations while ensuring that your wealth continues to grow sustainably. 

We have recently identified exclusive investment opportunities in the private markets that could align seamlessly with your strategic objectives. These selections are tailored for discerning clients like yourself and present unique prospects for both growth and stability.

I would appreciate the opportunity to discuss these developments in more detail and explore how they might fit within your current investme

## Part 1b — Three Agents in Parallel

**Upgrade:** instead of running one advisor at a time, we fire all three simultaneously using `asyncio.gather()`.

**New concept:** parallelism — all three agents run at the same time, results collected together. Faster, and gives us three different styles to compare.

In [10]:
# --- Run all three advisors in parallel and compare drafts ---
# asyncio.gather() fires all three simultaneously - faster than sequential runs.

message = "Write a HNW client outreach email"

with trace("Parallel HNW outreach drafts"):
    results = await asyncio.gather(
        Runner.run(advisor_agent1, message),
        Runner.run(advisor_agent2, message),
        Runner.run(advisor_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Elevate Your Investment Strategy with Apex Capital

Dear [Client's Name],

I hope this message finds you well.

As a valued client of Apex Capital, I wanted to take a moment to discuss the current landscape of wealth management and how our bespoke services can enhance your investment strategy.

In today's dynamic market environment, capital preservation has never been more critical. At Apex Capital, we recognize the distinct needs of ultra-high net worth individuals like yourself and are committed to providing tailored portfolio management solutions that align with your financial objectives and risk tolerance.

Our team of seasoned professionals leverages deep market insights and exclusive access to private market opportunities, ensuring that your portfolio is not only robust but also positioned to capitalize on emerging trends. We understand that each client is unique, and we are dedicated to curating personalized investment strategies that reflect your specific vision for we

## Part 1c — Add a Picker Agent (Agent as Evaluator)

**Upgrade:** a fourth agent (`prospect_picker`) receives all three drafts and selects the best one.

**New concept:** agent-as-evaluator — one agent judging the output of others, simulating the perspective of a real HNW prospect.

In [11]:
# --- A picker agent selects the strongest draft ---
# Evaluates from the perspective of a discerning HNW prospect.

prospect_picker = Agent(
    name="HNW Prospect Evaluator",
    instructions=(
        "You evaluate outreach emails as if you are a sophisticated, time-poor High Net Worth individual "
        "with significant investable assets. Pick the single email most likely to earn a reply - "
        "one that feels personal, credible, and relevant. "
        "Do not explain your choice; reply with the selected email only."
    ),
    model="gpt-4o-mini",
)

In [12]:
# --- Generate, evaluate, and print the best draft ---

message = "Write a HNW client outreach email"

with trace("HNW draft selection"):
    results = await asyncio.gather(
        Runner.run(advisor_agent1, message),
        Runner.run(advisor_agent2, message),
        Runner.run(advisor_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Outreach email drafts:\n\n" + "\n\nDraft:\n\n".join(outputs)

    best = await Runner.run(prospect_picker, emails)

    print(f"Best outreach email:\n{best.final_output}")

Best outreach email:
Subject: Navigating Opportunities in Today's Market

Dear [Client’s Name],

I hope this message finds you well. As we transition into the final quarter of the year, I wanted to reach out to discuss the current market landscape and how it may present unique opportunities for you.

Recent developments, such as [specific recent market trend or event], have created both challenges and avenues for growth in our investment strategies. I know that your primary goals include [client's likely financial goals, e.g., wealth preservation, growth, philanthropic initiatives]. With these in mind, I believe we can explore tailored solutions that align with your vision for the future.

At Apex Capital, we pride ourselves on our proactive approach to asset management and our commitment to protecting and enhancing your wealth. I would love the opportunity to discuss how we can navigate these currents together, ensuring your portfolio is positioned for both stability and growth.

Coul

Now check the trace:

https://platform.openai.com/traces

## Part 2 — Tools: `@function_tool` and `.as_tool()`

**Upgrade:** instead of just printing the winning draft, agents can now *act* — by calling tools.

**New concepts:**
- `@function_tool` — decorate any Python function to make it callable by an agent (SDK auto-generates JSON schema)
- `.as_tool()` — convert any agent into a tool so an orchestrator can call it like a function

In [13]:
# Recreate agents cleanly for the tools section

advisor_agent1 = Agent(
    name="Senior Private Banker",
    instructions=instructions1,
    model="gpt-4o-mini",
)

advisor_agent2 = Agent(
    name="Relationship Manager",
    instructions=instructions2,
    model="gpt-4o-mini",
)

advisor_agent3 = Agent(
    name="Managing Director",
    instructions=instructions3,
    model="gpt-4o-mini",
)

In [15]:
advisor_agent1

Agent(name='Senior Private Banker', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a Senior Private Banker at Apex Capital, an investment bank serving ultra-high net worth individuals. You write formal, authoritative outreach emails that emphasise capital preservation, bespoke portfolio management, and exclusive access to private market opportunities.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', res

## Steps 2 and 3: Tools and Agent Interactions

Simply wrap your function with `@function_tool` - the SDK generates all the JSON schema automatically.

remove sent_flag if multiple email sent

In [17]:
sent_flag = False

@function_tool
def send_email(body: str):
    """
    global sent_flag

    if sent_flag:
        return {"status": "blocked_duplicate_send"}

    sent_flag = True
    """    
    """ Send a plain-text outreach email to a HNW prospect """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("pashaajaz@gmail.com")  # Change to your verified sender
    to_email = To("pashaajaz@gmail.com")        # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Investment Opportunity - Apex Capital", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

### The function is now a tool - the SDK has generated the boilerplate JSON schema

In [18]:
send_email

FunctionTool(name='send_email', description='global sent_flag', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11b37de00>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)

### You can also convert an Agent into a tool

In [19]:
tool1 = advisor_agent1.as_tool(tool_name="advisor_agent1", tool_description="Write a HNW client outreach email")
tool1

FunctionTool(name='advisor_agent1', description='Write a HNW client outreach email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11b226c30>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)

### Gather all tools: one per advisor + the send_email function

In [20]:
description = "Write a HNW client outreach email mil"

tool1 = advisor_agent1.as_tool(tool_name="advisor_agent1", tool_description=description)
tool2 = advisor_agent2.as_tool(tool_name="advisor_agent2", tool_description=description)
tool3 = advisor_agent3.as_tool(tool_name="advisor_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='advisor_agent1', description='Write a HNW client outreach email mil', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11b3979b0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None),
 FunctionTool(name='advisor_agent2', description='Write a HNW client outreach email mil', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additio

## Part 3a — Orchestrator Agent (Tools Only)

**Upgrade:** a **Head of Private Banking** orchestrator agent is given all three advisor tools plus `send_email`.
It independently decides to call all three, evaluate the drafts, pick the winner, and send — no explicit sequential logic in our code.

**Key insight:** this is the line that crosses from a deterministic *workflow* into a true *agent* — the orchestrator decides its own steps at runtime.

It restricts to one call

In [21]:
head_of_pb_instructions = """
You are the Head of Private Banking.

You MUST follow this exact sequence:

1. Call advisor_agent1 ONCE
2. Call advisor_agent2 ONCE
3. Call advisor_agent3 ONCE

4. Select the best email immediately
5. Call send_email ONCE with the selected email

RULES:
- Do NOT call any tool more than once
- Do NOT regenerate drafts
- Do NOT revise or retry
- Do NOT send more than one email
- Once email is sent, stop immediately
"""

In [22]:
head_of_pb_instructions = """
You are the Head of Private Banking at Apex Capital.
Your goal is to send the single most effective outreach email to a High Net Worth prospect.

Follow these steps carefully:
1. Generate Drafts: Use all three advisor_agent tools to produce three different email drafts.
   Do not proceed until all three drafts are ready.

2. Evaluate and Select: Choose the draft most likely to resonate with a sophisticated HNW individual.
   Consider tone, credibility, and relevance. You may use the tools again if unsatisfied.

3. Send: Use the send_email tool to send ONLY the winning draft.

Crucial Rules:
- You must use the advisor_agent tools to draft emails - do not write them yourself.
- You must send exactly ONE email - never more than one.
"""




head_of_pb = Agent(
    name="Head of Private Banking",
    instructions=head_of_pb_instructions,
    tools=tools,
    model="gpt-4o-mini",
)

message = "Send an outreach email addressed to 'Dear Mr. Al-Rashid' from James Whitmore, Head of Private Banking"

with trace("HNW outreach - Head of Private Banking"):
    result = await Runner.run(head_of_pb, message)

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

## Part 3b — Handoffs + Email Operations Manager

**Final upgrade:** instead of sending plain text directly, the Head of PB now **hands off** the winning draft to a dedicated `Email Operations Manager`.
That agent uses two further sub-agents — a Subject Line Writer and an HTML Formatter — before sending the polished email.

**New concept:** handoffs vs tools
- **Tools** — control *returns* to the calling agent after the tool finishes
- **Handoffs** — control passes *across* to the receiving agent permanently

This creates a full specialised pipeline: draft → evaluate → format → send.

In [23]:
# --- Sub-agents for formatting and sending ---

subject_instructions = (
    "You write compelling email subject lines for HNW client outreach. "
    "The subject should feel exclusive and timely - not salesy. "
    "Return the subject line only, no explanation."
)

html_instructions = (
    "You convert a plain-text email body (which may contain markdown) into a clean HTML email. "
    "Use a simple, elegant layout befitting a premium investment bank. No flashy colours."
)

subject_writer = Agent(name="Subject Line Writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject line for a HNW outreach email")

html_converter = Agent(name="HTML Email Formatter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a plain-text email body to HTML")

In [24]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send a formatted HTML outreach email to a HNW prospect """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("pashaajaz@gmail.com")  # Change to your verified sender
    to_email = To("pashaajaz@gmail.com")        # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [25]:
tools = [subject_tool, html_tool, send_html_email]

In [26]:
tools

[FunctionTool(name='subject_writer', description='Write a subject line for a HNW outreach email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11b366b30>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None),
 FunctionTool(name='html_converter', description='Convert a plain-text email body to HTML', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object'

In [27]:
# --- Email Manager: formats and sends via handoff ---

emailer_instructions = (
    "You are an Email Operations Manager. You receive a draft email body. "
    "First, use the subject_writer tool to generate a subject line. "
    "Then, use the html_converter tool to convert the body to HTML. "
    "Finally, use the send_html_email tool to send the email with the subject and HTML body."
)

emailer_agent = Agent(
    name="Email Operations Manager",
    instructions=emailer_instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Format an email as HTML and send it to the prospect",
)

### Now we have 3 advisor tools + 1 handoff to the Email Operations Manager

In [28]:
description = "Write a HNW client outreach email"

tool1 = advisor_agent1.as_tool(tool_name="advisor_agent1", tool_description=description)
tool2 = advisor_agent2.as_tool(tool_name="advisor_agent2", tool_description=description)
tool3 = advisor_agent3.as_tool(tool_name="advisor_agent3", tool_description=description)

tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

print(tools)
print(handoffs)

[FunctionTool(name='advisor_agent1', description='Write a HNW client outreach email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11b398870>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None), FunctionTool(name='advisor_agent2', description='Write a HNW client outreach email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProper

In [29]:
# --- Head of Private Banking with handoff capability ---

head_of_pb_instructions = """
You are the Head of Private Banking at Apex Capital.
Your goal is to send the single most effective outreach email to a High Net Worth prospect.

Follow these steps carefully:
1. Generate Drafts: Use all three advisor_agent tools to produce three different email drafts.
   Do not proceed until all three drafts are ready.

2. Evaluate and Select: Choose the draft most likely to resonate with a sophisticated HNW individual.
   You may use the tools again if you're not satisfied with the initial results.

3. Handoff: Pass ONLY the winning draft to the 'Email Operations Manager' agent for formatting and sending.

Crucial Rules:
- You must use the advisor_agent tools to draft emails - do not write them yourself.
- You must hand off exactly ONE email - never more than one.
"""

head_of_pb = Agent(
    name="Head of Private Banking",
    instructions=head_of_pb_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini",
)

message = "Send an outreach email addressed to 'Dear Mr. Al-Rashid' from James Whitmore, Head of Private Banking"

with trace("HNW Outreach - Full Pipeline"):
    result = await Runner.run(head_of_pb, message)

### Check the trace and your inbox

https://platform.openai.com/traces

## Extra note: Google Agent Development Kit

Google's ADK follows a very similar pattern to the OpenAI Agents SDK:

```python
root_agent = Agent(
    name="portfolio_advisor",
    model="gemini-2.0-flash",
    description="Agent to answer questions about portfolio allocation and market conditions.",
    instruction="You are a helpful private banking agent advising HNW clients on investment strategy.",
    tools=[get_market_data, get_portfolio_summary]
)
```

Familiar, right? The patterns you've learned here transfer directly.